# 02 — OCI consumer to Bronze


In [ ]:
import base64, os, re
import oci
from pyspark.sql import Row, functions as F

catalog = os.getenv('AIDP_CATALOG','aidp_poc')
group = os.getenv('CONSUMER_GROUP','meter_bronze_consumer_v1')
partitions = tuple(value.strip() for value in os.getenv('OCI_STREAM_PARTITIONS','0').split(',') if value.strip())
missing = [key for key in ('OCI_STREAM_ENDPOINT','OCI_STREAM_OCID','OCI_CONFIG_FILE') if not os.getenv(key)]
if missing:
    raise ValueError('Missing required environment variables: ' + ', '.join(missing))
if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*',catalog) or not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*',group) or not partitions or any(not value.isdigit() for value in partitions):
    raise ValueError('Invalid catalog, consumer group, or partition list')
bronze = f'{catalog}.bronze.meter_reading'
checkpoints = f'{catalog}.bronze.stream_consumer_checkpoint'
client = oci.streaming.StreamClient(oci.config.from_file(os.environ['OCI_CONFIG_FILE'],os.getenv('OCI_PROFILE','DEFAULT')),service_endpoint=os.environ['OCI_STREAM_ENDPOINT'])

for partition in partitions:
    saved = spark.sql(f"SELECT last_offset FROM {checkpoints} WHERE consumer_group='{group}' AND stream_partition='{partition}' ORDER BY committed_at DESC LIMIT 1").collect()
    details = oci.streaming.models.CreateCursorDetails(type='AFTER_OFFSET',offset=saved[0].last_offset) if saved else oci.streaming.models.CreateCursorDetails(type='TRIM_HORIZON')
    cursor = client.create_cursor(os.environ['OCI_STREAM_OCID'],partition,details).data.value
    messages = client.get_messages(os.environ['OCI_STREAM_OCID'],cursor,limit=1000).data.messages
    rows = [Row(stream_partition=partition,stream_offset=str(message.offset),stream_key=base64.b64decode(message.key).decode() if message.key else None,raw_value=base64.b64decode(message.value).decode()) for message in messages]
    if not rows:
        print(f'No messages for partition {partition}')
        continue
    spark.createDataFrame(rows).withColumn('ingested_at',F.current_timestamp()).withColumn('payload_json',F.col('raw_value')).createOrReplaceTempView('bronze_batch')
    spark.sql(f'MERGE INTO {bronze} t USING bronze_batch s ON t.stream_partition=s.stream_partition AND t.stream_offset=s.stream_offset WHEN NOT MATCHED THEN INSERT *')
    spark.createDataFrame([(group,partition,rows[-1].stream_offset)],'consumer_group string, stream_partition string, last_offset string').withColumn('committed_at',F.current_timestamp()).createOrReplaceTempView('checkpoint_batch')
    spark.sql(f'INSERT INTO {checkpoints} SELECT * FROM checkpoint_batch')
    print(f'Processed {len(rows)} messages from partition {partition}')